# TD03 — Perceptron multi-couche et rétropropagation

## Objectif

Un perceptron (TD01) puis une couche mono-couche (TD02) ne séparent que des données **linéairement séparables**. Ici on **empile des couches** et on apprend les poids par **rétropropagation du gradient** (cours « 4 - Rétropropagation »).

On procède en deux temps :
1. Implémenter un **perceptron à une couche cachée** (2 couches), le **valider** avec des fonctions de test, et lui faire apprendre **XOR**.
2. **Généraliser** à un MLP à $n$ couches, puis lui faire apprendre des **fonctions** en 1D et 2D (régression), avec **split train/test** et **early stopping**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

# Alias de types pour les tableaux numpy
Vec = np.ndarray[tuple[int], np.dtype[np.float64]]        # vecteur 1D de flottants
Mat = np.ndarray[tuple[int, int], np.dtype[np.float64]]   # matrice 2D de flottants

## 1. Perceptron à une couche cachée

### 1.1 Fonctions d'activation

Le cours utilise une activation générique $A$. Pour la rétropropagation, on a besoin de $A$ **et** de sa dérivée $A'$ :

- sigmoïde : $\sigma(z) = \frac{1}{1+e^{-z}}$, $\sigma'(z) = \sigma(z)(1-\sigma(z))$
- tangente hyperbolique : $\tanh'(z) = 1 - \tanh^2(z)$

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 1</legend>

Complétez `sigmoid_prime` et `tanh_prime`.

</fieldset>

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z: Mat) -> Mat:
    """
    σ'(z) = σ(z) · (1 - σ(z)).
    """
    # BEGIN
    # raise NotImplementedError
    s = sigmoid(z)
    return s * (1.0 - s)
    # END


def tanh_prime(z: Mat) -> Mat:
    """
    tanh'(z) = 1 - tanh²(z).
    """
    # BEGIN
    # raise NotImplementedError
    return 1.0 - np.tanh(z) ** 2
    # END

In [ ]:
def check_activations():
    assert sigmoid_prime(np.array([[0.0]]))[0, 0] == 0.25, "sigmoid_prime(0) doit valoir 0.25"
    assert tanh_prime(np.array([[0.0]]))[0, 0] == 1.0, "tanh_prime(0) doit valoir 1"
    z = np.array([[-2.0, -0.5], [0.3, 1.7]])
    eps = 1e-6
    num_sig = (sigmoid(z + eps) - sigmoid(z - eps)) / (2 * eps)
    num_tanh = (np.tanh(z + eps) - np.tanh(z - eps)) / (2 * eps)
    assert sigmoid_prime(z).shape == z.shape and tanh_prime(z).shape == z.shape, "la forme doit être conservée"
    assert np.allclose(sigmoid_prime(z), num_sig, atol=1e-8), "sigmoid_prime incorrecte"
    assert np.allclose(tanh_prime(z), num_tanh, atol=1e-8), "tanh_prime incorrecte"
    print("activations check — OK")


check_activations()

In [ ]:
ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh": (np.tanh, tanh_prime),
    "linear": (lambda z: z, lambda z: np.ones_like(z)),
}

### 1.2 Le modèle

On commence par le réseau le plus simple possible qui sache faire XOR : **une seule couche cachée** (2 couches de poids). C'est le cas « deux couches de neurones » du cours (§2.2.2).

Convention **ligne** : chaque échantillon est une ligne de la matrice. Pour un batch $X$ de forme $(N, d)$ :

#### Notations

| Symbole | Signification |
|---|---|
| $N$ | nombre d'échantillons du batch |
| $d$ | nombre de features en entrée |
| $h$ | nombre de neurones de la couche cachée |
| $k$ | nombre de neurones de sortie |
| $X$ | batch d'entrée, de forme $(N, d)$ |
| $W_1$ | matrice de poids de la couche cachée, de forme $(h, d)$ |
| $b_1$ | biais de la couche cachée, de forme $(h,)$ |
| $W_2$ | matrice de poids de la couche de sortie, de forme $(k, h)$ |
| $b_2$ | biais de la couche de sortie, de forme $(k,)$ |
| $Z_1$ | pré-activations de la couche cachée |
| $Y_1$ | activations de la couche cachée, de forme $(N, h)$ |
| $Z_2$ | pré-activations de la couche de sortie |
| $Y_2$ | sortie du réseau, $Y = Y_2$, de forme $(N, k)$ |
| $A$ | fonction d'activation de la couche cachée (`activation`) ; $A'$ sa dérivée |
| $A_o$ | fonction d'activation de la couche de sortie (`output_activation`) ; $A_o'$ sa dérivée |
| $T$ | cibles, de même forme que $Y$ |
| $E$ | coût (erreur quadratique moyenne) |
| $\delta_1, \delta_2$ | erreurs rétropropagées de la couche cachée et de la couche de sortie |
| $\odot$ | produit terme à terme (Hadamard) |

#### Passe avant

$$Z_1 = X W_1^T + b_1, \qquad Y_1 = A(Z_1), \qquad Z_2 = Y_1 W_2^T + b_2, \qquad Y = Y_2 = A_o(Z_2)$$

#### Coût

$$E = \tfrac12\,\operatorname{mean}\big((Y - T)^2\big)$$

#### Passe arrière

$$\delta_2 = (Y_2 - T)\odot A_o'(Z_2), \qquad \frac{\partial E}{\partial W_2} = \tfrac{1}{N}\,\delta_2^T Y_1, \qquad \frac{\partial E}{\partial b_2} = \tfrac{1}{N}\sum_i \delta_2$$

$$\delta_1 = (\delta_2\, W_2)\odot A'(Z_1), \qquad \frac{\partial E}{\partial W_1} = \tfrac{1}{N}\,\delta_1^T X, \qquad \frac{\partial E}{\partial b_1} = \tfrac{1}{N}\sum_i \delta_1$$

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 2</legend>

Complétez les méthodes de `PerceptronCache` :

1. `forward` : renvoie $(Y_0, Z_1, Y_1, Z_2, Y_2)$, avec $Y_0 = X$ ;
2. `loss` : coût $E$ (utiliser `predict`) ;
3. `backward` : renvoie `(dW1, db1), (dW2, db2)` ;
4. `_step` : une étape de descente de gradient, $W \leftarrow W - \eta\,\partial E / \partial W$ (idem pour les biais) ;
5. `fit` : descente de gradient par lot, une étape `_step` par epoch ; renvoie l'historique du coût (une valeur par epoch, calculée **après** l'étape).

</fieldset>

In [ ]:
class PerceptronCache:
    def __init__(self, n_in, n_hidden, n_out, activation="sigmoid", output_activation="sigmoid", seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0.0, 0.5, size=(n_hidden, n_in))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(0.0, 0.5, size=(n_out, n_hidden))
        self.b2 = np.zeros(n_out)
        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.output_activation, self.output_activation_prime = ACTIVATIONS[output_activation]


    def predict(self, X):
        return self.forward(X)[-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût du modèle : erreur quadratique moyenne.

        E = (1/2) · mean(Σ_k (Y_k - T_k)²).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        float
            Coût quadratique moyen sur le batch.
        """
        # BEGIN
        # raise NotImplementedError
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))
        # END

    def forward(self, X: Mat) -> tuple[Mat, Mat, Mat, Mat, Mat]:
        """Passe avant du perceptron à une couche cachée.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).

        Returns
        -------
        tuple[Mat, Mat, Mat, Mat, Mat]
            (Y0, Z1, Y1, Z2, Y2) : entrée, pré-activations et activations
            de la couche cachée puis de la couche de sortie.
        """
        # BEGIN
        # raise NotImplementedError
        Y0 = X
        Z1 = Y0 @ self.W1.T + self.b1
        Y1 = self.activation(Z1)
        Z2 = Y1 @ self.W2.T + self.b2
        Y2 = self.output_activation(Z2)
        return Y0, Z1, Y1, Z2, Y2
        # END

    
    def backward(self, X: Mat, t: Mat) -> tuple[tuple[Mat, Vec], tuple[Mat, Vec]]:
        """Passe arrière : gradients du coût par rapport aux paramètres.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        tuple[tuple[Mat, Vec], tuple[Mat, Vec]]
            ((dW1, db1), (dW2, db2)) : gradients pour chaque couche.
        """
        # BEGIN
        # raise NotImplementedError
        Y0, Z1, Y1, Z2, Y2 = self.forward(X)
        N = X.shape[0]
        delta2 = (Y2 - t) * self.output_activation_prime(Z2)
        dW2 = delta2.T @ Y1 / N
        db2 = delta2.sum(axis=0) / N
        delta1 = (delta2 @ self.W2) * self.activation_prime(Z1)
        dW1 = delta1.T @ Y0 / N
        db1 = delta1.sum(axis=0) / N
        return (dW1, db1), (dW2, db2)
        # END

    def _step(self, X: Mat, t: Mat, lr: float) -> None:
        """Effectue une étape de descente de gradient (mise à jour des poids).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float
            Pas d'apprentissage.
        """
        # BEGIN
        # raise NotImplementedError
        (dW1, db1), (dW2, db2) = self.backward(X, t)
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        # END
    
    def fit(self, X: Mat, t: Mat, lr: float = 0.5, epochs: int = 10000) -> list[float]:
        """Entraîne le réseau par descente de gradient

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float, optionnel
            Pas d'apprentissage (défaut : 0.5).
        epochs : int, optionnel
            Nombre d'epochs (défaut : 10000).

        Returns
        -------
        list[float]
            Historique du coût, un élément par epoch.
        """
        # BEGIN
        # raise NotImplementedError
        history = []
        for epoch in range(epochs):
            self._step(X, t, lr)
            history.append(self.loss(X, t))
        return history
        # END

#### Vérification de `PerceptronCache`

On **valide** les méthodes avant de croire que le réseau marche. `forward`, `loss`, `backward` et `_step` sont comparés à des valeurs de référence **pré-calculées** (sur un réseau et un batch fixés) ; `fit` est comparé à une suite d'appels à `_step`.

##### `forward`

Sur un réseau et un batch fixés, on compare $Z_1, Y_1, Z_2, Y_2$ (et `predict`) à des valeurs de référence pré-calculées.

In [ ]:
def check_forward(model, X, tol=1e-8):
    Y0, Z1, Y1, Z2, Y2 = model.forward(X)
    assert Y0.shape == X.shape, "Y0 doit être X"
    assert np.allclose(Y0, X, atol=tol), "Y0 != X"
    Z1r = np.array([
        [-0.032544688, 0.106849667, 0.027431171],
        [0.153753892, 0.037459021, 0.313317574],
        [0.055987095, -0.324110116, -0.161782286],
        [0.614390352, -0.401652722, 0.801668475],
    ])
    Y1r = np.array([
        [0.491864546, 0.526687031, 0.506857363],
        [0.538362927, 0.509363660, 0.577694836],
        [0.513993119, 0.419674409, 0.459642415],
        [0.648941649, 0.400915321, 0.690331271],
    ])
    Z2r = np.array([[0.113372657], [0.108768160], [0.067731533], [0.068101779]])
    Y2r = np.array([[0.528312844], [0.527165264], [0.516926413], [0.517018868]])
    assert Z1.shape == Z1r.shape, "forme Z1 incorrecte"
    assert Y1.shape == Y1r.shape, "forme Y1 incorrecte"
    assert Z2.shape == Z2r.shape, "forme Z2 incorrecte"
    assert Y2.shape == Y2r.shape, "forme Y2 incorrecte"
    assert np.allclose(Z1, Z1r, atol=tol), "Z1 incorrect"
    assert np.allclose(Y1, Y1r, atol=tol), "Y1 incorrect"
    assert np.allclose(Z2, Z2r, atol=tol), "Z2 incorrect"
    assert np.allclose(Y2, Y2r, atol=tol), "Y2 incorrect"
    assert np.allclose(model.predict(X), Y2r, atol=tol), "predict != forward"
    print("forward check — OK")


m_check = PerceptronCache(2, 3, 1, seed=1)
Xc = np.random.default_rng(0).normal(size=(4, 2))
check_forward(m_check, Xc)

##### `loss` et `backward`

`backward` est facile à coder faux sans le voir. On compare le coût et les gradients renvoyés à des valeurs de référence pré-calculées.

In [ ]:
def check_loss(model, X, t, tol=1e-8):
    assert abs(model.loss(X, t) - 0.433488946) < tol, f"loss incorrecte : {model.loss(X, t)}"
    print("loss check — OK")


def check_gradients(model, X, t, tol=1e-8):
    (dW1, db1), (dW2, db2) = model.backward(X, t)

    dW1r = np.array([
        [-0.007956438, -0.006635153],
        [0.009149589, 0.007559421],
        [0.005034272, 0.004230065],
    ])
    db1r = np.array([-0.007270309, 0.008245889, 0.004657691])
    dW2r = np.array([[0.075443684, 0.047091093, 0.078962968]])
    db2r = np.array([0.118306372])

    assert dW1.shape == dW1r.shape, "forme dW1 incorrecte"
    assert db1.shape == db1r.shape, "forme db1 incorrecte"
    assert dW2.shape == dW2r.shape, "forme dW2 incorrecte"
    assert db2.shape == db2r.shape, "forme db2 incorrecte"
    assert np.allclose(dW1, dW1r, atol=tol), "dW1 incorrect"
    assert np.allclose(db1, db1r, atol=tol), "db1 incorrect"
    assert np.allclose(dW2, dW2r, atol=tol), "dW2 incorrect"
    assert np.allclose(db2, db2r, atol=tol), "db2 incorrect"
    print("gradient check — OK")


m_check = PerceptronCache(2, 3, 1, seed=1)
Xc = np.random.default_rng(0).normal(size=(4, 2))
tc = np.random.default_rng(1).normal(size=(4, 1))
check_loss(m_check, Xc, tc)
check_gradients(m_check, Xc, tc)

##### `_step` et `fit`

`_step` applique une descente de gradient : on vérifie les poids obtenus après **une** étape (réseau, batch et `lr` fixés). `fit` sur 2 epochs doit donner exactement les mêmes poids que 2 appels à `_step`, et un historique de 2 coûts.

In [ ]:
def check_step(model, X, t, lr, tol=1e-8):
    model._step(X, t, lr)

    W1r = np.array([
        [0.176770315, 0.414126648],
        [0.160643744, -0.655358326],
        [0.450160797, 0.221072254],
    ])
    b1r = np.array([0.003635154, -0.004122944, -0.002328845])
    W2r = np.array([[-0.306198460, 0.267013506, 0.142804714]])
    b2r = np.array([-0.059153186])

    assert np.allclose(model.W1, W1r, atol=tol), f"W1 incorrect : {model.W1}"
    assert np.allclose(model.b1, b1r, atol=tol), f"b1 incorrect : {model.b1}"
    assert np.allclose(model.W2, W2r, atol=tol), f"W2 incorrect : {model.W2}"
    assert np.allclose(model.b2, b2r, atol=tol), f"b2 incorrect : {model.b2}"
    print("step check — OK")


m_check = PerceptronCache(2, 3, 1, seed=1)
Xc = np.random.default_rng(0).normal(size=(4, 2))
tc = np.random.default_rng(1).normal(size=(4, 1))
check_step(m_check, Xc, tc, lr=0.5)


def check_fit(X, t, lr):
    m_fit = PerceptronCache(2, 3, 1, seed=1)
    m_ref = PerceptronCache(2, 3, 1, seed=1)
    history = m_fit.fit(X, t, lr=lr, epochs=2)
    losses = []
    for _ in range(2):
        m_ref._step(X, t, lr)
        losses.append(m_ref.loss(X, t))
    for name in ("W1", "b1", "W2", "b2"):
        assert np.array_equal(getattr(m_fit, name), getattr(m_ref, name)), f"{name} incorrect après fit"
    assert history == losses, f"historique attendu {losses}, obtenu {history}"
    print("fit check — OK")


check_fit(Xc, tc, lr=0.5)

### 1.3 XOR

Le XOR n'est pas linéairement séparable (TD01). Un perceptron à **une couche cachée** le résout : entrée de taille 2, couche cachée de taille 4, sortie sigmoïde (on classe 0/1, seuil 0.5).

La cellule suivante entraîne le réseau et affiche ses 4 prédictions ; vérifiez qu'elles sont correctes.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
t_xor = np.array([[0], [1], [1], [0]], dtype=float)

model_xor = PerceptronCache(2, 4, 1, activation="sigmoid", output_activation="sigmoid", seed=0)
model_xor.fit(X_xor, t_xor, lr=1.0, epochs=20000)

pred = model_xor.predict(X_xor).ravel()
print("sorties  :", pred.round(3).tolist())
print("prédiction (seuil 0.5) :", (pred > 0.5).astype(int).tolist())
print("cibles   :", t_xor.ravel().tolist())
assert ((pred > 0.5).astype(int) == t_xor.ravel()).all(), "XOR mal appris"

La frontière de décision est la courbe où la sortie vaut 0.5.

In [ ]:
def plot_decision_2d(model, X, t, lim=(-0.5, 1.5)):
    xx, yy = np.meshgrid(np.linspace(lim[0], lim[1], 200), np.linspace(lim[0], lim[1], 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    fig, ax = plt.subplots()
    ax.contourf(xx, yy, Z, levels=np.linspace(0, 1, 21), cmap="RdBu")
    ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=2)
    ax.scatter(X[:, 0], X[:, 1], c=t.ravel(), edgecolors="k", s=80, cmap="RdBu", vmin=0, vmax=1, zorder=3)
    ax.set_aspect("equal")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")
    ax.grid(True)

plot_decision_2d(model_xor, X_xor, t_xor)

## 2. MLP à $n$ couches

### 2.1 Généralisation

Le perceptron à une couche cachée est le cas $L = 2$. Pour apprendre des fonctions plus complexes, on **empile** $L$ couches. La classe générique `MLP(sizes)` prend `sizes=[d, h1, …, k]` pour décrire l'architecture.

Convention **ligne** : chaque échantillon est une ligne de la matrice. Pour un batch $X$ de taille $(N, d)$ :

#### Notations

| Symbole | Signification |
|---|---|
| $N$ | nombre d'échantillons du batch |
| $d$ | nombre de features en entrée |
| $L$ | nombre de couches |
| $l$ | indice de couche, $l = 1, \dots, L$ |
| $X$ | batch d'entrée, de forme $(N, d)$ |
| $Y_0$ | activations de la couche d'entrée ($Y_0 = X$) |
| $W_l$ | matrice de poids de la couche $l$, de forme $(n_{out}, n_{in})$ |
| $b_l$ | biais de la couche $l$, de forme $(n_{out},)$ |
| $Z_l$ | pré-activations de la couche $l$ (avant activation) |
| $Y_l$ | activations de la couche $l$, de forme $(N, n_{out})$ |
| $A$ | fonction d'activation des couches cachées (`activation`) ; $A'$ sa dérivée |
| $A_o$ | fonction d'activation de la couche de sortie (`output_activation`) ; $A_o'$ sa dérivée |
| $Y$ | sortie du réseau, $Y = Y_L$ |
| $T$ | cibles, de même forme que $Y$ |
| $E$ | coût (erreur quadratique moyenne) |
| $\delta_l$ | erreur rétropropagée, $\delta_l = \frac{\partial E}{\partial Z_l}$ |
| $\odot$ | produit terme à terme (Hadamard) |

#### Passe avant

$$Y_0 = X,\qquad Z_l = Y_{l-1} W_l^T + b_l,\qquad Y_l = A(Z_l) \;\; (l < L),\qquad Y = Y_L = A_o(Z_L)$$

#### Coût

$$E = \tfrac12\,\operatorname{mean}\big((Y - T)^2\big)$$

#### Passe arrière

$$\delta_L = (Y - T)\odot A_o'(Z_L)$$

$$\delta_l = (\delta_{l+1}\,W_{l+1})\odot A'(Z_l)$$

$$\frac{\partial E}{\partial W_l} = \frac{1}{N}\,\delta_l^T Y_{l-1},\qquad \frac{\partial E}{\partial b_l} = \frac{1}{N}\sum_i \delta_l$$

Les couches cachées utilisent `activation` (fonction $A$) ; la couche de sortie utilise `output_activation` (fonction $A_o$).

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 3</legend>

Complétez les méthodes de `MLP` :

1. `forward` : boucle sur les couches ; renvoie `(As, Zs)`, avec `As = [Y_0, Y_1, …, Y_L]` et `Zs = [Z_1, …, Z_L]` ;
2. `loss` : coût $E$ (comme pour `PerceptronCache`) ;
3. `backward` : boucle des couches de sortie vers l'entrée ; renvoie la liste `[(dW_1, db_1), …, (dW_L, db_L)]` ;
4. `step` : une étape de descente de gradient sur toutes les couches ;
5. `fit_batch` : descente de gradient par lot simple, une étape `step` par epoch ; renvoie l'historique du coût `{"train": [...]}`, avec en plus une clé `"test"` si `X_test`/`t_test` sont fournis (coûts calculés **après** chaque étape).

</fieldset>

In [ ]:
class MLP:
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws = []
        self.bs = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))
        self.activation, self.activation_prime = ACTIVATIONS[activation]
        self.output_activation, self.output_activation_prime = ACTIVATIONS[output_activation]

    def predict(self, X):
        return self.forward(X)[0][-1]

    def loss(self, X: Mat, t: Mat) -> float:
        """Coût du modèle : erreur quadratique moyenne.

        E = (1/2) · mean(Σ_k (Y_k - T_k)²).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        float
            Coût quadratique moyen sur le batch.
        """
        # coût du cours : moyenne sur les échantillons de 1/2 · Σ_k (Y_k - T_k)^2
        # BEGIN
        # raise NotImplementedError
        return 0.5 * float(np.mean(np.sum((self.predict(X) - t) ** 2, axis=1)))
        # END



    def forward(self, X: Mat) -> tuple[list[Mat], list[Mat]]:
        """Passe avant du MLP, couche par couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).

        Returns
        -------
        tuple[list[Mat], list[Mat]]
            (As, Zs) : activations (avec Y0 = X en tête) et pré-activations
            de chaque couche.
        """
        # BEGIN
        # raise NotImplementedError
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            f = self.output_activation if l == n - 1 else self.activation
            A = f(Z)
            As.append(A)
        return As, Zs
        # END

    def backward(self, X: Mat, t: Mat) -> list[tuple[Mat, Vec]]:
        """Passe arrière : gradients du coût pour chaque couche.

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).

        Returns
        -------
        list[tuple[Mat, Vec]]
            Liste de (dW_l, db_l), une paire de gradients par couche.
        """
        # BEGIN
        # raise NotImplementedError
        As, Zs = self.forward(X)
        N = X.shape[0]
        n = len(self.Ws)
        delta = (As[-1] - t) * self.output_activation_prime(Zs[-1])
        dWs, dbs = [], []
        for l in range(n - 1, -1, -1):
            dW = delta.T @ As[l] / N
            db = delta.sum(axis=0) / N
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if l > 0:
                delta = (delta @ self.Ws[l]) * self.activation_prime(Zs[l - 1])
        return list(zip(dWs, dbs))
        # END

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Effectue une étape de descente de gradient (mise à jour des poids).

        Parameters
        ----------
        X : Mat
            Batch d'entrée, de forme (N, d).
        t : Mat
            Cibles, de même forme que la sortie du réseau (N, k).
        lr : float
            Pas d'apprentissage.
        """
        # BEGIN
        # raise NotImplementedError
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.Ws[l] -= lr * dW
            self.bs[l] -= lr * db
        # END

    def fit_batch(self, X: Mat, t: Mat, X_test: Mat | None = None, t_test: Mat | None = None, lr: float = 0.1, epochs: int = 2000) -> dict[str, list[float]]:
        """Entraîne le MLP par descente de gradient par lot.

        Si `X_test`/`t_test` sont fournis, le coût de test est aussi
        ajouté à l'historique.

        Parameters
        ----------
        X : Mat
            Batch d'entraînement, de forme (N, d).
        t : Mat
            Cibles d'entraînement, de forme (N, k).
        X_test : Mat, optionnel
            Batch de test, de forme (N_test, d).
        t_test : Mat, optionnel
            Cibles de test, de forme (N_test, k).
        lr : float, optionnel
            Pas d'apprentissage (défaut : 0.1).
        epochs : int, optionnel
            Nombre maximal d'epochs (défaut : 2000).

        Returns
        -------
        dict[str, list[float]]
            Historique des coûts `{"train": [...], "test": [...]}`.
        """
        # BEGIN
        # raise NotImplementedError
        history = {"train": []}
        if X_test is not None:
            history["test"] = []
        for epoch in range(epochs):
            self.step(X, t, lr)
            history["train"].append(self.loss(X, t))
            if X_test is not None:
                history["test"].append(self.loss(X_test, t_test))
        return history
        # END


#### Vérification de `MLP`

- `forward`, `backward`, `step` : un `MLP([2, 3, 1])` sigmoïde, de graine 1, a exactement les mêmes poids que le `PerceptronCache(2, 3, 1, seed=1)` déjà validé : on compare leurs passes avant et arrière, puis on vérifie les poids après **une** étape (valeurs pré-calculées).
- `fit_batch` : sur un réseau linéaire à une couche, la simulation à la main d'une epoch donne des valeurs exactes.

In [ ]:
def check_forward_backward_mlp():
    m = MLP([2, 3, 1], activation="sigmoid", output_activation="sigmoid", seed=1)
    p = PerceptronCache(2, 3, 1, seed=1)
    X = np.random.default_rng(0).normal(size=(4, 2))
    t = np.random.default_rng(1).normal(size=(4, 1))
    As, Zs = m.forward(X)
    Y0, Z1, Y1, Z2, Y2 = p.forward(X)
    assert len(As) == 3 and len(Zs) == 2, "As doit contenir L+1 éléments (avec X), Zs L éléments"
    for name, a, b in [("As[0]", As[0], Y0), ("Zs[0]", Zs[0], Z1), ("As[1]", As[1], Y1), ("Zs[1]", Zs[1], Z2), ("As[2]", As[2], Y2)]:
        assert a.shape == b.shape and np.allclose(a, b), f"{name} incorrect"
    assert np.isclose(m.loss(X, t), p.loss(X, t)), "loss incorrecte"
    grads = m.backward(X, t)
    (dW1, db1), (dW2, db2) = p.backward(X, t)
    assert len(grads) == 2, "backward doit renvoyer une paire (dW, db) par couche"
    for name, a, b in [("dW_1", grads[0][0], dW1), ("db_1", grads[0][1], db1), ("dW_2", grads[1][0], dW2), ("db_2", grads[1][1], db2)]:
        assert a.shape == b.shape and np.allclose(a, b), f"{name} incorrect"
    print("forward/backward (MLP) check — OK")


def check_step_mlp(model, X, t, lr, tol=1e-8):
    model.step(X, t, lr)

    W0r = np.array([
        [0.172054568, 0.410844400],
        [0.166012500, -0.651611420],
        [0.453171081, 0.223162969],
    ])
    b0r = np.array([0.000213575, -0.000210689, -0.000153188])
    W1r = np.array([[-0.269613257, 0.289535450, 0.181938749]])
    b1r = np.array([-0.003118110])

    assert np.allclose(model.Ws[0], W0r, atol=tol), f"Ws[0] incorrect : {model.Ws[0]}"
    assert np.allclose(model.bs[0], b0r, atol=tol), f"bs[0] incorrect : {model.bs[0]}"
    assert np.allclose(model.Ws[1], W1r, atol=tol), f"Ws[1] incorrect : {model.Ws[1]}"
    assert np.allclose(model.bs[1], b1r, atol=tol), f"bs[1] incorrect : {model.bs[1]}"
    print("step (MLP) check — OK")


m_check = MLP([2, 3, 1], activation="sigmoid", output_activation="sigmoid", seed=1)
Xc = np.random.default_rng(0).normal(size=(3, 2))
tc = np.random.default_rng(1).normal(size=(3, 1))
check_step_mlp(m_check, Xc, tc, lr=0.5)


def check_fit_batch():
    model = MLP([2, 1], output_activation="linear", seed=0)
    model.Ws = [np.array([[1.0, -2.0]])]
    model.bs = [np.array([0.0])]
    X = np.array([[1.0, 0.0], [0.0, 1.0]])
    t = np.array([[0.0], [0.0]])
    hist = model.fit_batch(X, t, X_test=X, t_test=t, lr=0.5, epochs=1)

    # simulation à la main (sortie linéaire) :
    #   z = X @ W.T + b = [[1], [-2]]  ;  delta = z - t = [[1], [-2]]
    #   dW = delta.T @ X / 2 = [[0.5, -1]]  ;  db = delta.sum(0) / 2 = -0.5
    #   W -= lr*dW -> [[0.75, -1.5]]  ;  b -= lr*db -> 0.25
    #   coût après l'étape : z = [[1], [-1.25]] -> E = 0.5 * mean([1, 1.5625]) = 0.640625
    assert np.array_equal(model.Ws[0], np.array([[0.75, -1.5]])), model.Ws[0]
    assert np.array_equal(model.bs[0], np.array([0.25])), model.bs[0]
    assert hist == {"train": [0.640625], "test": [0.640625]}, hist
    assert "test" not in MLP([2, 1]).fit_batch(X, t, epochs=1), "pas de clé 'test' sans jeu de test"
    print("fit_batch check — OK")


check_forward_backward_mlp()
check_fit_batch()

### 2.2 Fonction 1D : apprendre $f(x) = \sin(x)$

Régression cette fois : sortie **linéaire** (pas de sigmoïde), données bruitées, séparation **train/test**.

In [ ]:
def gen_1d(n=300, noise=0.3, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-np.pi, np.pi, size=(n, 1))
    t = np.sin(X) + rng.normal(0.0, noise, size=(n, 1))
    return X, t

X1, t1 = gen_1d()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4a</legend>

1. Séparez aléatoirement `X1, t1` en train (`X_train, t_train`, 60 points, tirés avec `np.random.default_rng(0)`) et test (`X_test, t_test`, le reste).
2. Entraînez un réseau `model_1d = MLP([1, 32, 32, 1], activation="tanh", output_activation="linear", seed=0)` avec `fit_batch` (`lr=0.05`, `epochs=50000`) en passant le jeu de test ; stockez l'historique dans `hist_1d`.

</fieldset>

In [ ]:
# BEGIN
## Indice : rng.permutation(len(X1)) donne un ordre aléatoire des indices
# X_train, t_train = ...
# X_test, t_test = ...
# model_1d = ...
# hist_1d = ...
rng = np.random.default_rng(0)
idx = rng.permutation(len(X1))
n_train = 60
X_train, t_train = X1[idx[:n_train]], t1[idx[:n_train]]
X_test, t_test = X1[idx[n_train:]], t1[idx[n_train:]]

model_1d = MLP([1, 32, 32, 1], activation="tanh", output_activation="linear", seed=0)
hist_1d = model_1d.fit_batch(X_train, t_train, X_test, t_test, lr=0.05, epochs=50000)
# END

In [ ]:
def check_split_1d():
    assert X_train.shape == (60, 1) and t_train.shape == (60, 1), "train : 60 points attendus"
    assert X_test.shape == (240, 1) and t_test.shape == (240, 1), "test : 240 points attendus"
    X_all = np.concatenate([X_train, X_test])
    t_all = np.concatenate([t_train, t_test])
    order, order_ref = np.argsort(X_all.ravel()), np.argsort(X1.ravel())
    assert np.array_equal(X_all[order], X1[order_ref]) and np.array_equal(t_all[order], t1[order_ref]), \
        "train et test doivent former une partition de X1, t1 (couples (x, t) conservés)"
    assert len(hist_1d["train"]) == 50000 and len(hist_1d["test"]) == 50000, "historique de 50000 epochs attendu"
    assert hist_1d["test"][-1] == model_1d.loss(X_test, t_test), "hist_1d['test'] doit être calculé sur le jeu de test"
    print("split + entraînement 1D check — OK")


check_split_1d()

L'historique `hist_1d` contient à la fois le coût d'entraînement et le coût de test ; la cellule ci-dessous le trace (fonction `plot_history`).

In [ ]:
def plot_history(history, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.plot(history["train"], label="train")
    if history["test"]:
        ax.plot(history["test"], label="test")
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE")
    ax.set_yscale("log")
    ax.legend()
    ax.grid(True)
    return ax

plot_history(hist_1d)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4b</legend>

Que font les erreurs train et test au fil des epochs ? Observez-vous du **surapprentissage** (le test remonte alors que le train continue de baisser) ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

Le coût train décroît pendant toute la durée de l'entraînement. Le coût test décroît d'abord comme lui, atteint son minimum vers l'epoch 1000 (≈ 0.045), puis **remonte** lentement (≈ 0.051 à la fin) alors que le train continue de baisser (≈ 0.033).

C'est du **surapprentissage** : avec seulement 60 points bruités et un réseau de ~1100 paramètres, le réseau finit par apprendre le bruit des données d'entraînement, ce qui dégrade la prédiction sur des points nouveaux.

</fieldset>

`fit_batch` entraîne un nombre d'epochs fixe : rien n'empêche le réseau de **surapprendre** le train (le coût train baisse encore alors que le coût test remonte). L'**early stopping** arrête l'apprentissage quand la performance sur le test se dégrade.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4c</legend>

Complétez `fit_early_stopping` (ajoutée à `MLP`) : à chaque epoch, on fait une étape `step`, puis on évalue le coût sur le train et sur le **test** (ajoutés à l'historique). On garde les meilleurs poids (coût test minimal) ; on s'arrête dès que `patience` epochs consécutives se sont écoulées sans amélioration, puis on restaure les meilleurs poids.

</fieldset>

In [ ]:
def fit_early_stopping(self, X: Mat, t: Mat, X_test: Mat, t_test: Mat, lr: float = 0.1, epochs: int = 2000, patience: int = 200) -> dict[str, list[float]]:
    """Entraîne le MLP par descente de gradient par lot avec early stopping.

    À chaque epoch on évalue le coût sur le jeu de test ; on conserve les
    meilleurs poids ; on s'arrête après `patience` epochs sans amélioration.

    Returns
    -------
    dict[str, list[float]]
        Historique des coûts `{"train": [...], "test": [...]}`.
    """
    # BEGIN
    # raise NotImplementedError
    ## Indice : copy.deepcopy(self.Ws) pour sauvegarder les poids
    history = {"train": [], "test": []}
    best_test = np.inf
    best = None
    no_improve = 0
    for epoch in range(epochs):
        self.step(X, t, lr)
        history["train"].append(self.loss(X, t))
        te = self.loss(X_test, t_test)
        history["test"].append(te)
        if te < best_test:
            best_test = te
            best = (copy.deepcopy(self.Ws), copy.deepcopy(self.bs))
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    if best is not None:
        self.Ws, self.bs = best
    return history
    # END


MLP.fit_early_stopping = fit_early_stopping


In [ ]:
def check_fit_early_stopping():
    model = MLP([1, 1], output_activation="linear", seed=0)
    model.Ws = [np.array([[0.0]])]
    model.bs = [np.array([0.0])]
    X = np.array([[1.0]])
    hist = model.fit_early_stopping(X, np.array([[2.0]]), X, np.array([[1.0]]), lr=0.25, epochs=100, patience=2)
    # simulation à la main (z = w + b, dW = db = z - 2) :
    #   epoch 1 : z=0    -> w=b=0.5   -> z=1    : train 0.5,     test 0       (meilleur)
    #   epoch 2 : z=1    -> w=b=0.75  -> z=1.5  : train 0.125,   test 0.125   (1 sans amélioration)
    #   epoch 3 : z=1.5  -> w=b=0.875 -> z=1.75 : train 0.03125, test 0.28125 (2 -> arrêt)
    #   restauration des poids de l'epoch 1
    assert hist == {"train": [0.5, 0.125, 0.03125], "test": [0.0, 0.125, 0.28125]}, hist
    assert np.array_equal(model.Ws[0], np.array([[0.5]])), f"meilleurs poids non restaurés : {model.Ws[0]}"
    assert np.array_equal(model.bs[0], np.array([0.5])), f"meilleurs biais non restaurés : {model.bs[0]}"
    print("fit_early_stopping check — OK")


check_fit_early_stopping()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4d</legend>

Entraînez un **nouveau** réseau `model_es`, de même architecture et même graine que `model_1d`, avec `fit_early_stopping` (`lr=0.05`, `epochs=50000`, `patience=4000`) sur le même split ; stockez l'historique dans `hist_es`.

</fieldset>

In [ ]:
# BEGIN
# model_es = ...
# hist_es = ...
model_es = MLP([1, 32, 32, 1], activation="tanh", output_activation="linear", seed=0)
hist_es = model_es.fit_early_stopping(X_train, t_train, X_test, t_test, lr=0.05, epochs=50000, patience=4000)
# END

In [ ]:
def check_es():
    n = len(hist_es["test"])
    best = int(np.argmin(hist_es["test"]))
    assert n == best + 1 + 4000, "l'entraînement doit s'arrêter 4000 epochs après le meilleur coût test"
    assert model_es.loss(X_test, t_test) == hist_es["test"][best], "model_es doit avoir les meilleurs poids"
    print("early stopping 1D check — OK")


check_es()

Après la vérification, la cellule suivante trace l'historique `hist_es`.

In [ ]:
plot_history(hist_es)

Comparons la prédiction du réseau `model_es` à la vraie fonction $\sin(x)$.

In [ ]:
def plot_fit_1d(model, X, t, ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    grid = np.linspace(-np.pi, np.pi, 300).reshape(-1, 1)
    ax.plot(grid.ravel(), np.sin(grid).ravel(), label="sin(x)", color="tab:green", lw=2)
    ax.plot(grid.ravel(), model.predict(grid).ravel(), label="prédiction", color="tab:red", lw=2, ls="--")
    ax.scatter(X.ravel(), t.ravel(), s=15, alpha=0.5, label="données (train)", color="tab:blue")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(); ax.grid(True)
    return ax

plot_fit_1d(model_es, X_train, t_train)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 4e</legend>

Comparez l'historique `hist_es` à `hist_1d` : que font les erreurs train et test ? À quoi sert l'early stopping ici ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

Les deux historiques sont identiques jusqu'à l'arrêt (même graine, mêmes données) : `hist_es` s'interrompt vers l'epoch 5000, soit `patience` = 4000 epochs après le minimum du coût test (epoch ≈ 1000), au lieu de continuer jusqu'à 50000.

Le coût train final est un peu plus élevé qu'avec `fit_batch`, mais les poids restaurés sont ceux du **minimum du coût test** : le réseau ne surapprend pas, et on économise ~90 % du temps de calcul. L'early stopping sert de **régularisation** : il arrête l'apprentissage au moment où le réseau commence à apprendre le bruit.

</fieldset>

### 2.3 Fonction 2D : $f(x_1, x_2) = \sin(x_1)\cos(x_2)$

Même principe, deux entrées. On visualise la surface prédite en 3D avec **plotly**.

In [ ]:
def gen_2d(n=500, noise=0.1, seed=1):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-3.0, 3.0, size=(n, 2))
    t = (np.sin(X[:, 0]) * np.cos(X[:, 1])).reshape(-1, 1) + rng.normal(0.0, noise, size=(n, 1))
    return X, t

X2, t2 = gen_2d()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 5</legend>

1. Séparez `X2, t2` en train (`X2_train, t2_train`, 400 points, tirés avec `np.random.default_rng(1)`) et test (`X2_test, t2_test`, le reste).
2. Entraînez `model_2d = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)` avec `fit_early_stopping` (`lr=0.2`, `epochs=30000`, `patience=2000`) ; stockez l'historique dans `hist_2d`.

</fieldset>

In [ ]:
# BEGIN
# X2_train, t2_train = ...
# X2_test, t2_test = ...
# model_2d = ...
# hist_2d = ...
rng = np.random.default_rng(1)
idx = rng.permutation(len(X2))
n_train = 400
X2_train, t2_train = X2[idx[:n_train]], t2[idx[:n_train]]
X2_test, t2_test = X2[idx[n_train:]], t2[idx[n_train:]]

model_2d = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
hist_2d = model_2d.fit_early_stopping(X2_train, t2_train, X2_test, t2_test, lr=0.2, epochs=30000, patience=2000)
# END

In [ ]:
def check_2d():
    assert X2_train.shape == (400, 2) and X2_test.shape == (100, 2), "400 points de train et 100 de test attendus"
    assert t2_train.shape == (400, 1) and t2_test.shape == (100, 1)
    assert model_2d.sizes == [2, 16, 16, 1], "architecture [2, 16, 16, 1] attendue"
    assert model_2d.loss(X2_test, t2_test) == min(hist_2d["test"]), "model_2d doit avoir les meilleurs poids"
    print("entraînement 2D check — OK")


check_2d()

Après la vérification, la cellule suivante affiche la surface vraie et la surface prédite.

In [ ]:
import plotly.graph_objects as go

def plot_surface_3d(model, x1, x2):
    xx, yy = np.meshgrid(x1, x2)
    G = np.c_[xx.ravel(), yy.ravel()]
    Z_pred = model.predict(G).reshape(xx.shape)
    Z_true = np.sin(xx) * np.cos(yy)
    fig = go.Figure()
    fig.add_trace(go.Surface(x=x1, y=x2, z=Z_true, colorscale="Viridis", opacity=0.9, name="vraie", showscale=False))
    fig.add_trace(go.Surface(x=x1, y=x2, z=Z_pred, colorscale="Reds", opacity=0.9, name="prédite", showscale=False))
    fig.update_layout(title="Fonction 2D : vraie (vert) vs prédite (rouge)", height=600,
                      scene=dict(xaxis_title="x1", yaxis_title="x2", zaxis_title="f"))
    fig.show()

x1 = np.linspace(-3, 3, 50)
x2 = np.linspace(-3, 3, 50)
plot_surface_3d(model_2d, x1, x2)

### 2.4 Élan (momentum)

La descente de gradient par lot (sections 2.2 et 2.3) avance à petits pas, surtout dans les vallées étroites de la surface d'erreur où elle **zigzague**. L'**élan** (momentum) ajoute une « inertie » : au lieu de déplacer $W$ selon le gradient brut, on déplace selon une **vitesse** $v$ qui mélange le gradient courant et la vitesse précédente.

$$v_l \leftarrow \mu\, v_l + \eta\, \frac{\partial E}{\partial W_l}, \qquad W_l \leftarrow W_l - v_l$$

- $\mu$ : coefficient d'élan, $0 \le \mu < 1$ (typiquement $0.9$). $\mu = 0$ redonne la descente de gradient classique.
- $\eta$ : pas d'apprentissage (`lr`).
- $v_l$ : vitesse associée à $W_l$, même forme que $W_l$, initialisée à $0$.

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6a</legend>

Complétez `MLPMomentum`, sous-classe de `MLP` : elle stocke des vitesses `vWs`/`vbs` (mêmes formes que `Ws`/`bs`, nulles au départ) et **surcharge** `step` pour appliquer la règle ci-dessus. La classe de base garde sa méthode `step` **sans élan**.

</fieldset>

In [ ]:
class MLPMomentum(MLP):
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0, momentum=0.9):
        super().__init__(sizes, activation, output_activation, seed)
        self.momentum = momentum
        self.vWs = [np.zeros_like(W) for W in self.Ws]
        self.vbs = [np.zeros_like(b) for b in self.bs]

    def step(self, X: Mat, t: Mat, lr: float) -> None:
        """Une étape de descente de gradient avec élan (momentum).

        v_l <- mu * v_l + lr * dW_l ; W_l <- W_l - v_l.
        """
        # BEGIN
        # raise NotImplementedError
        for l, (dW, db) in enumerate(self.backward(X, t)):
            self.vWs[l] = self.momentum * self.vWs[l] + lr * dW
            self.vbs[l] = self.momentum * self.vbs[l] + lr * db
            self.Ws[l] -= self.vWs[l]
            self.bs[l] -= self.vbs[l]
        # END

In [ ]:
def check_momentum():
    model = MLPMomentum([2, 1], output_activation="linear", seed=0, momentum=0.5)
    model.Ws = [np.array([[1.0, -2.0]])]
    model.bs = [np.array([0.0])]
    model.vWs = [np.zeros((1, 2))]
    model.vbs = [np.zeros(1)]
    X = np.array([[1.0, 0.0], [0.0, 1.0]])
    t = np.array([[0.0], [0.0]])
    # simulation à la main (lr = 0.5, mu = 0.5) :
    #   étape 1 : dW = [[0.5, -1]], db = -0.5 -> v = [[0.25, -0.5]], vb = -0.25
    #             W = [[0.75, -1.5]], b = 0.25
    #   étape 2 : z = [[1], [-1.25]] -> dW = [[0.5, -0.625]], db = -0.125
    #             v = 0.5 v + 0.5 dW = [[0.375, -0.5625]], vb = -0.1875
    #             W = [[0.375, -0.9375]], b = 0.4375
    model.step(X, t, lr=0.5)
    assert np.array_equal(model.Ws[0], np.array([[0.75, -1.5]])), f"étape 1 : {model.Ws[0]}"
    assert np.array_equal(model.bs[0], np.array([0.25])), f"étape 1 : {model.bs[0]}"
    model.step(X, t, lr=0.5)
    assert np.array_equal(model.vWs[0], np.array([[0.375, -0.5625]])), f"vitesse : {model.vWs[0]}"
    assert np.array_equal(model.Ws[0], np.array([[0.375, -0.9375]])), f"étape 2 : {model.Ws[0]}"
    assert np.array_equal(model.bs[0], np.array([0.4375])), f"étape 2 : {model.bs[0]}"
    print("momentum check — OK")


check_momentum()

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6b</legend>

Entraînez le problème 2D (`X2_train, t2_train`) **sans** élan (`model_plain`, un `MLP`) et **avec** élan $\mu = 0.9$ (`model_mom`, un `MLPMomentum`), avec `fit_batch` : architecture `[2, 16, 16, 1]`, `tanh`/`linear`, `seed=0`, `lr=0.2`, `epochs=20000` pour les deux. Stockez les historiques dans `hist_plain` et `hist_mom`.

</fieldset>

In [ ]:
# BEGIN
# model_plain = ...
# hist_plain = ...
# model_mom = ...
# hist_mom = ...
lr = 0.2
epochs = 20000

model_plain = MLP([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0)
hist_plain = model_plain.fit_batch(X2_train, t2_train, lr=lr, epochs=epochs)

model_mom = MLPMomentum([2, 16, 16, 1], activation="tanh", output_activation="linear", seed=0, momentum=0.9)
hist_mom = model_mom.fit_batch(X2_train, t2_train, lr=lr, epochs=epochs)
# END

In [ ]:
def check_plain_mom():
    assert type(model_plain) is MLP and isinstance(model_mom, MLPMomentum), "model_plain : MLP, model_mom : MLPMomentum"
    assert model_mom.momentum == 0.9, "μ = 0.9 attendu"
    assert len(hist_plain["train"]) == 20000 and len(hist_mom["train"]) == 20000, "20000 epochs attendues"
    assert hist_plain["train"][-1] == model_plain.loss(X2_train, t2_train)
    assert hist_mom["train"][-1] == model_mom.loss(X2_train, t2_train)
    print("élan check — OK")


check_plain_mom()

Après la vérification, la cellule suivante trace les deux courbes de coût.

In [ ]:
fig, ax = plt.subplots()
ax.plot(hist_plain["train"], label="sans élan (μ=0)")
ax.plot(hist_mom["train"], label="avec élan (μ=0.9)")
ax.set_xlabel("epoch"); ax.set_ylabel("MSE")
ax.set_yscale("log"); ax.legend(); ax.grid(True)

<fieldset style="border: 3px solid #1976d2; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #1976d2; font-weight: bold; font-size: 1.5em; padding: 0 6px">✏️ Exercice 6c</legend>

Comparez les deux courbes de coût : quel est l'effet de l'élan ?

</fieldset>

*Votre réponse :*

<!-- TOCLEAR -->
<fieldset style="border: 3px solid #2e7d32; border-radius: 8px; padding: 0 14px; margin: 12px 0">
<legend style="color: #2e7d32; font-weight: bold; font-size: 1.5em; padding: 0 6px">💡 Réponse</legend>

Avec élan, le coût baisse **plus vite** : à chaque instant de l'entraînement il est nettement en dessous de celui sans élan (par ex. ≈ 0.0065 contre ≈ 0.0105 à l'epoch 1000), et le coût final est plus bas. L'inertie accumule les gradients qui vont dans la même direction (accélération dans les vallées) et atténue les zigzags.

En contrepartie, la courbe avec élan est moins régulière : l'inertie peut faire **dépasser** un minimum et remonter un peu le coût (oscillations visibles en fin d'entraînement).

</fieldset>

## Bonus

- **Perceptron à une couche cachée** : que se passe-t-il si on prend 2 ou 3 neurones cachés pour XOR au lieu de 4 ?
- **Activations** : comparez sigmoïde vs tanh sur XOR (vitesse, nombre d'epochs).
- **Early stopping** : réglez `patience` très faible puis très élevée — où s'arrête l'apprentissage ?
- **Comparaison d'architectures** : entraînez plusieurs réseaux (ex. `[1, 8, 8, 1]` vs `[1, 64, 64, 1]`, ou sigmoïde vs tanh) sur la fonction 1D, tracez leurs courbes train/test sur un même graphe : qui surapprend ? qui converge le plus vite ?
- **Autres fonctions** : testez d'autres fonctions (ex. $f(x) = x^2$, $f(x_1,x_2) = x_1^2 + x_2^2$) ou plus de bruit : où le réseau échoue-t-il ?